In [ ]:
# 1. Clone repository & Install base requirements
!git clone https://github.com/doantrongthai/flood_mamba_edge
%cd flood_mamba_edge
!pip install -r requirements.txt


In [ ]:
# 2. Setup Official Hardware-Accelerated Mamba CUDA Kernel on Google Colab GPU (Tesla T4)
import zipfile, json, sys, sysconfig, pathlib, subprocess, torch

SP = pathlib.Path(sysconfig.get_paths()["purelib"])
FILE_ID = "1kuBAKlNtIc8G1KQFqO8xV7t9ZEM4XATB"

assert torch.cuda.get_device_capability(0) == (7, 5), f"Vui long chon Colab GPU la Tesla T4 (capability 7.5), hien tai: {torch.cuda.get_device_name(0)}"

# 1) Tai goi pre-compiled CUDA wheel
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"])
import gdown
gdown.download(id=FILE_ID, output="/content/pack.zip", quiet=False)

# 2) Giai nen vao site-packages
with zipfile.ZipFile("/content/pack.zip") as z:
    assert z.testzip() is None, "File tai ve bi hong, vui long tai lai"
    m = json.loads(z.read("__manifest__.json"))
    now = {"python": sys.version.split()[0], "torch": torch.__version__, "cuda": torch.version.cuda}
    bad = {k: (m[k], now[k]) for k in now if m[k] != now[k]}
    assert not bad, f"Moi truong Colab khong khop: {bad}"
    for name in z.namelist():
        if not name.startswith("__"):
            z.extract(name, SP)
print("Da giai nen vao:", SP)

# 3) Cai cac dependency bo tro
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "packaging", "ninja", "transformers", "triton"])

# 4) Kiem tra CUDA kernel Mamba tren GPU
import selective_scan_cuda, causal_conv1d_cuda
from mamba_ssm.modules.mamba_simple import causal_conv1d_fn
from mamba_ssm import Mamba
print("causal_conv1d_fn:", "OK (CUDA kernel)" if causal_conv1d_fn is not None else "FALLBACK")
y = Mamba(d_model=64).cuda()(torch.randn(2, 128, 64, device="cuda"))
y.mean().backward()
print("[OK] OFFICIAL MAMBA HARDWARE CUDA KERNEL ACTIVE:", y.shape)


In [ ]:
import os

# ==============================================================================
# SELECT MODEL ARCHITECTURE (Chi can thay doi MODEL_NAME o day):
# Options: "lcmamnet", "mambafloodlite", "lightm_unet", "satreg_cmunet", "ultralight_vmunet", "hybridseg"
# ==============================================================================
MODEL_NAME = "lcmamnet"

AUTO_PRETRAINED_MODELS = {'deeplabv3'}
IS_PRETRAINED = MODEL_NAME.lower() in AUTO_PRETRAINED_MODELS
PRETRAINED_FLAG = "--pretrained" if IS_PRETRAINED else ""

IMAGE_SIZE = 128
BATCH_SIZE = 4
EPOCHS = 100
LR = 0.001
LOSS_NAME = "lovasz_focal"
NUM_CLASSES = 1

MULTISEED = True
SEEDS = [42, 123, 456, 789, 2024]
REP_SEED = 42

def get_checkpoint_path(model, loss, dataset, seed, is_multiseed):
    clean_loss = loss.lower().replace('-', '_')
    candidates = [
        f"outputs/seed_{seed}/{model}_{clean_loss}_{dataset}_s{seed}.pth",
        f"outputs/{model}_{clean_loss}_{dataset}_s{seed}.pth",
        f"outputs/seed_{seed}/{model}_{loss}_{dataset}_s{seed}.pth",
        f"outputs/{model}_{loss}_{dataset}_s{seed}.pth"
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return candidates[0] if is_multiseed else candidates[1]


In [ ]:
# 3. Model Profiling & Edge Complexity Analysis (FP32 vs FP16)
import torch
from models import get_model
from utils.metrics import calculate_model_complexity, measure_inference_time

print("=" * 70)
print(f"MAMBA EDGE ARCHITECTURE PROFILING: {MODEL_NAME.upper()}")
print("=" * 70)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = get_model(MODEL_NAME, num_classes=NUM_CLASSES, input_size=IMAGE_SIZE).to(device)
model.eval()

complexity = calculate_model_complexity(model, input_size=(1, 3, IMAGE_SIZE, IMAGE_SIZE), device=device)
inference = measure_inference_time(model, input_size=(1, 3, IMAGE_SIZE, IMAGE_SIZE), device=device)

dep_params = complexity.get('deployable_params', complexity['total_params'])
dep_size_mb = complexity.get('deployable_size_mb', complexity['model_size_mb'])
gmacs_val = complexity.get('gmacs', complexity['gflops'] / 2)

print(f"Total Params (Deployable) : {dep_params:,}")
print(f"Model Storage Size (FP32) : {dep_size_mb:.2f} MB")
print(f"Model Storage Size (FP16) : {dep_size_mb / 2:.2f} MB (Edge Half-Precision)")
print(f"GMACs                     : {gmacs_val:.4f}")
print(f"GFLOPs                    : {complexity['gflops']:.4f}")
print(f"Edge Latency (GPU T4)     : {inference['latency_ms']:.2f} ms")
print(f"Throughput (FPS)          : {inference['fps']:.2f} FPS")
print("=" * 70)


In [ ]:
# 4. Multi-Seed Training on FloodKaggle
if MULTISEED:
    seeds_str = " ".join(map(str, SEEDS))
    !python main.py --dataset floodkaggle --model {MODEL_NAME} --num_classes {NUM_CLASSES} --loss {LOSS_NAME} --size {IMAGE_SIZE} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --multiseed --seeds {seeds_str} {PRETRAINED_FLAG} --download
else:
    !python main.py --dataset floodkaggle --model {MODEL_NAME} --num_classes {NUM_CLASSES} --loss {LOSS_NAME} --size {IMAGE_SIZE} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --seed {REP_SEED} {PRETRAINED_FLAG} --download


In [ ]:
# 5. Export Deployable ONNX Model (FloodKaggle) - NO INT8
weights_fk = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "floodkaggle", REP_SEED, MULTISEED)
!python export.py --model {MODEL_NAME} --weights {weights_fk} --size {IMAGE_SIZE} --num_classes {NUM_CLASSES} --output outputs/{MODEL_NAME}_floodkaggle_deploy.onnx


In [ ]:
# 6. Multi-Seed Training on AIFloodSense
if MULTISEED:
    seeds_str = " ".join(map(str, SEEDS))
    !python main.py --dataset aifloodsense --model {MODEL_NAME} --num_classes {NUM_CLASSES} --loss {LOSS_NAME} --size {IMAGE_SIZE} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --multiseed --seeds {seeds_str} {PRETRAINED_FLAG} --download
else:
    !python main.py --dataset aifloodsense --model {MODEL_NAME} --num_classes {NUM_CLASSES} --loss {LOSS_NAME} --size {IMAGE_SIZE} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --seed {REP_SEED} {PRETRAINED_FLAG} --download


In [ ]:
# 7. Export Deployable ONNX Model (AIFloodSense) - NO INT8
weights_afs = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "aifloodsense", REP_SEED, MULTISEED)
!python export.py --model {MODEL_NAME} --weights {weights_afs} --size {IMAGE_SIZE} --num_classes {NUM_CLASSES} --output outputs/{MODEL_NAME}_aifloodsense_deploy.onnx


In [ ]:
# 8. Cross-Dataset Generalization: Source = FloodKaggle -> Targets = [FloodKaggle, AIFloodSense, FloodVN]
weights_fk = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "floodkaggle", REP_SEED, MULTISEED)
!python cross_dataset_eval.py --model {MODEL_NAME} --weights {weights_fk} --source_dataset floodkaggle --target_datasets floodkaggle aifloodsense floodvn --size {IMAGE_SIZE} --num_classes {NUM_CLASSES} --download


In [ ]:
# 9. Cross-Dataset Generalization: Source = AIFloodSense -> Targets = [AIFloodSense, FloodKaggle, FloodVN]
weights_afs = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "aifloodsense", REP_SEED, MULTISEED)
!python cross_dataset_eval.py --model {MODEL_NAME} --weights {weights_afs} --source_dataset aifloodsense --target_datasets aifloodsense floodkaggle floodvn --size {IMAGE_SIZE} --num_classes {NUM_CLASSES} --download


In [ ]:
# 10. Bundle All Artifacts & Download .zip package
import os
import shutil
from google.colab import files

bundle_dir = f"{MODEL_NAME}_All_Artifacts"
os.makedirs(bundle_dir, exist_ok=True)
output_dir = "outputs"

if os.path.exists(output_dir):
    for item in sorted(os.listdir(output_dir)):
        src = os.path.join(output_dir, item)
        dst = os.path.join(bundle_dir, item)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        elif os.path.isfile(src):
            shutil.copy2(src, dst)

zip_filename = f"{MODEL_NAME}_artifacts.zip"
if os.path.exists(zip_filename):
    os.remove(zip_filename)

shutil.make_archive(f"{MODEL_NAME}_artifacts", "zip", bundle_dir)
print(f"[OK] Artifacts compressed successfully: {zip_filename}")
files.download(zip_filename)
